### Load Data

In [1]:
import os
import numpy as np

# ✅ Path to your saved .npy file
npy_file_path = "../MediaPipe_landmarks/squat_back_new_landmarks.npy"

# ✅ Load the NumPy array
landmarks_data = np.load(npy_file_path)

# ✅ Check its shape
print("Shape:", landmarks_data.shape)

# ✅ Inspect first frame
print("First frame landmarks:\n", landmarks_data)


Shape: (742, 33, 3)
First frame landmarks:
 [[[ 0.45064944  0.36396754  0.17328146]
  [ 0.44843626  0.35616517  0.16817467]
  [ 0.44696623  0.35604924  0.1681011 ]
  ...
  [ 0.46197635  0.79510498 -0.02590076]
  [ 0.4363803   0.79401076 -0.01820946]
  [ 0.47002825  0.79102546 -0.05297839]]

 [[ 0.4506503   0.36336634  0.16061236]
  [ 0.44833249  0.35552183  0.15250959]
  [ 0.4468579   0.3553212   0.15244085]
  ...
  [ 0.46014279  0.79514694 -0.02586933]
  [ 0.43671101  0.79308593 -0.00983211]
  [ 0.47017464  0.79106367 -0.05263109]]

 [[ 0.4505659   0.36140192  0.1524386 ]
  [ 0.44801071  0.35407662  0.14435907]
  [ 0.44644299  0.35404211  0.14429715]
  ...
  [ 0.45985621  0.79499245 -0.02241618]
  [ 0.4367345   0.79298121 -0.00654933]
  [ 0.47018316  0.7910651  -0.04587119]]

 ...

 [[ 0.47704986  0.39920992  0.20848788]
  [ 0.47515395  0.39309323  0.20082073]
  [ 0.47373477  0.39288527  0.2007487 ]
  ...
  [ 0.48691025  0.78535026 -0.10753024]
  [ 0.43673748  0.76754844 -0.0392955 ]


### Index different joints and normalize skeleton to be centered by the pelvis and size of entire skeleton 

In [2]:
import sys, os

# Go two levels up to project root
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../"))
sys.path.append(project_root)

print("Project root added to path:", project_root)

from Utils.utils.utils import *



# MediaPipe joint indices
HIP_L = 23
KNEE_L = 25
ANKLE_L = 27
TOE_L = 31
HEEL_L = 29

HIP_R = 24
KNEE_R = 26
ANKLE_R = 28
TOE_R = 32
HEEL_R = 30

SHOULDER_L = 11
ELBOW_L = 13
WRIST_L = 15
THUMB_L = 21
INDEXFINGER_L = 19
PINKY_L = 17

SHOULDER_R = 12
ELBOW_R = 14
WRIST_R = 16
THUMB_R = 22
INDEXFINGER_R = 20
PINKY_R = 18

NOSE = 0



def normalize_skeleton_with_virtual_joints(coords, lhip_idx, rhip_idx,
                                           lsho_idx, rsho_idx, eps=1e-8):
    """
    coords: (T, J, 3) raw 3D landmarks
    lhip_idx, rhip_idx: left/right hip indices
    lsho_idx, rsho_idx: left/right shoulder indices

    Returns:
      coords_norm: (T, J+2, 3) normalized coords including virtual pelvis and neck
      pelvis:      (T, 3) pelvis positions before centering
      neck:        (T, 3) neck positions before centering
    """

    T, J, _ = coords.shape

    # 1) Virtual mid-pelvis and mid-shoulder (neck proxy)
    left_hip  = coords[:, lhip_idx, :]    # (T, 3)
    right_hip = coords[:, rhip_idx, :]    # (T, 3)
    pelvis = (left_hip + right_hip) / 2.0 # (T, 3)

    left_sho  = coords[:, lsho_idx, :]    # (T, 3)
    right_sho = coords[:, rsho_idx, :]    # (T, 3)
    neck = (left_sho + right_sho) / 2.0   # (T, 3)

    # 2) Center all original joints on pelvis
    coords_centered = coords - pelvis[:, None, :]  # (T, J, 3)

    # 3) Also center virtual joints
    pelvis_centered = pelvis - pelvis              # becomes (T, 3) at origin
    neck_centered   = neck - pelvis                # neck relative to pelvis

    # 4) Stack virtual joints at the end: [J joints, pelvis, neck]
    pelvis_centered = pelvis_centered[:, None, :]  # (T, 1, 3)
    neck_centered   = neck_centered[:, None, :]    # (T, 1, 3)
    coords_with_virtual = np.concatenate(
        [coords_centered, pelvis_centered, neck_centered], axis=1
    )  # (T, J+2, 3)

    # 5) Compute scale as pelvis->neck distance
    # neck is last joint index: J+1
    neck_rel = coords_with_virtual[:, -1, :]                # (T, 3)
    scale = np.linalg.norm(neck_rel, axis=-1, keepdims=True) + eps  # (T, 1)

    # 6) Scale all joints
    coords_norm = coords_with_virtual / scale[:, None, :]   # (T, J+2, 3)

    return coords_norm, pelvis, neck



coords_norm, pelvis_raw, neck_raw = normalize_skeleton_with_virtual_joints(
    landmarks_data, HIP_L, HIP_R, SHOULDER_L, SHOULDER_R
)

Project root added to path: c:\Users\chris\OneDrive\Desktop\Fritidsprojekt\TempAISpotter\AI\OlympicAi


### Transform landmark coordinates into angles of different joints. We use angles for our embedding

### Do i need to change to angles from anatomical to flexion?

In [3]:
def compute_angle_features(landmarks):
    frames, joints, dims = landmarks.shape
    features = []

    for f in range(frames):
        lm = landmarks[f]

        # Example angles
        left_ankle = calculate_angle(lm[KNEE_L], lm[ANKLE_L], lm[TOE_L])
        right_ankle = calculate_angle(lm[KNEE_R], lm[ANKLE_R], lm[TOE_R])
        
        left_knee = calculate_angle(lm[HIP_L], lm[KNEE_L], lm[ANKLE_L])
        right_knee = calculate_angle(lm[HIP_R], lm[KNEE_R], lm[ANKLE_R])
        
        left_hip = calculate_angle(lm[SHOULDER_L], lm[HIP_L], lm[KNEE_L])
        right_hip = calculate_angle(lm[SHOULDER_R], lm[HIP_R], lm[KNEE_R])
        
        left_shoulder = calculate_angle(lm[ELBOW_L], lm[SHOULDER_L], lm[HIP_L])
        right_shoulder = calculate_angle(lm[ELBOW_R], lm[SHOULDER_R], lm[HIP_R])

        left_elbow = calculate_angle(lm[SHOULDER_L], lm[ELBOW_L], lm[WRIST_L])
        right_elbow = calculate_angle(lm[SHOULDER_R], lm[ELBOW_R], lm[WRIST_R])
        
        left_wrist = calculate_angle(lm[ELBOW_L], lm[WRIST_L], lm[PINKY_L])
        right_wrist = calculate_angle(lm[ELBOW_R], lm[WRIST_R], lm[PINKY_R])

        # Add more angles if you want a richer embedding

        features.append([
            left_ankle,
            right_ankle,
            left_knee,
            right_knee,
            left_hip,
            right_hip,
            left_shoulder,
            right_shoulder,
            left_wrist,
            right_wrist,
            right_elbow,
            left_elbow,
        ])

    return np.array(features)

new_arr = compute_angle_features(landmarks_data)
new_arr.shape

(742, 12)

In [4]:
new_arr

array([[173.39960894, 165.07162933, 177.86105137, ..., 175.28570405,
        173.35736401, 169.41352812],
       [168.07339765, 151.69591624, 178.35217047, ..., 175.04697073,
        172.13746591, 174.26792544],
       [166.4135722 , 149.08536345, 178.61431974, ..., 173.89313418,
        172.38953449, 173.38387565],
       ...,
       [163.66823372, 142.02937216, 176.35302207, ..., 166.09049465,
         27.16563099,  32.75494832],
       [163.37072059, 142.30591565, 176.28641892, ..., 167.78985549,
         27.49737783,  32.6695756 ],
       [163.02060063, 142.55722534, 176.2132976 , ..., 167.98647707,
         27.77427081,  32.59752549]])

In [5]:
new_arr.shape

(742, 12)

### Normalize values from pure angles to (something that i need to check what it gets turned into)

In [6]:
#norm_arr = (new_arr - new_arr.mean(axis=0)) / new_arr.std(axis=0)
#norm_arr

### Add a feature which tells the velocity of movement. Used to compare speed of reps

In [7]:
velocity = np.diff(new_arr, axis=0)
velocity

array([[-5.32621129e+00, -1.33757131e+01,  4.91119090e-01, ...,
        -2.38733324e-01, -1.21989810e+00,  4.85439731e+00],
       [-1.65982545e+00, -2.61055278e+00,  2.62149272e-01, ...,
        -1.15383656e+00,  2.52068578e-01, -8.84049788e-01],
       [-1.72932950e+00, -1.90876907e+00,  1.15682357e-01, ...,
        -6.15236014e-01,  2.17187166e-01,  9.41739139e-01],
       ...,
       [-1.95443873e-01,  3.53120718e-01, -2.43590468e-02, ...,
         3.17768808e+00,  9.00460535e-02, -8.80482483e-03],
       [-2.97513127e-01,  2.76543490e-01, -6.66031510e-02, ...,
         1.69936085e+00,  3.31746839e-01, -8.53727132e-02],
       [-3.50119958e-01,  2.51309690e-01, -7.31213221e-02, ...,
         1.96621578e-01,  2.76892983e-01, -7.20501137e-02]])

### Add feature of knee and hip symmetry

In [8]:
left_knee  = new_arr[:, 3]
right_knee = new_arr[:, 4]
left_hip   = new_arr[:, 5]
right_hip  = new_arr[:, 6]



knee_symmetry = left_knee - right_knee
hip_symmetry  = left_hip - right_hip
knee_symmetry.shape, hip_symmetry.shape

((742,), (742,))

### Smoothed the angles to create less noise in the data. (Maybe this is only relevant if we feed it through a ML pipeline?)

In [9]:
from scipy.signal import savgol_filter
angles_smooth = savgol_filter(new_arr, window_length=11, polyorder=3, axis=0)
angles_smooth

array([[174.10647642, 163.48916988, 177.90296683, ..., 175.7071387 ,
        173.00769117, 170.45172101],
       [168.33471865, 154.73355934, 178.32625343, ..., 174.47513227,
        172.83661303, 172.30949565],
       [165.00433547, 148.97627021, 178.5481822 , ..., 173.71109441,
        172.51943822, 173.66838997],
       ...,
       [163.69600007, 141.98620901, 176.33688618, ..., 164.88148515,
         27.23229989,  32.72893436],
       [163.40872481, 142.28959339, 176.28255586, ..., 166.82987488,
         27.41799446,  32.67966694],
       [162.98784932, 142.583329  , 176.21967686, ..., 168.9162699 ,
         27.786638  ,  32.60182865]])

### Make a new embedding with the new features we created

In [10]:
min_frames = velocity.shape[0]  # 415

# Remove last frames to make vectors match in dimensions
angles_smooth_trimmed = angles_smooth[:min_frames]
knee_symmetry_trimmed = knee_symmetry[:min_frames]
hip_symmetry_trimmed  = hip_symmetry[:min_frames]

# Add dimension for concatenation
knee_symmetry_trimmed = knee_symmetry_trimmed.reshape(-1, 1)
hip_symmetry_trimmed  = hip_symmetry_trimmed.reshape(-1, 1)


velocity.shape, angles_smooth_trimmed.shape, knee_symmetry_trimmed.shape, hip_symmetry_trimmed.shape

embedding = np.concatenate([
    angles_smooth_trimmed,
    velocity,                  # already 415
    knee_symmetry_trimmed,
    hip_symmetry_trimmed
], axis=1)

embedding.shape

(741, 26)

In [11]:
np.save("../embedding/squat_back_new_embedding.npy", embedding)

In [12]:
bob = np.load("../embedding/squat_back_new_embedding.npy")
bobby = np.load("../embedding/squat_back_res_bob_angle_embedding_not_norm.npy")

In [13]:
bob.shape, bobby.shape

((741, 26), (415, 26))

### Use DTW to compare similarity of videos (figure out if i should use cosine, euclidean or manhatten distance metric)

In [ ]:
from dtw import dtw
from scipy.spatial.distance import cosine, euclidean
dist, cost, acc, path = dtw(bob, bobby, dist=lambda x, y: euclidean(x, y))  # Cosine similarity per frame
dist, path

(128017.3768645038,
 (array([  0,   1,   2,   3,   4,   5,   6,   7,   8,   9,  10,  11,  12,
          13,  14,  15,  16,  17,  18,  19,  20,  21,  22,  23,  24,  25,
          26,  27,  28,  29,  30,  31,  32,  33,  34,  35,  36,  37,  38,
          39,  40,  41,  42,  43,  44,  45,  46,  47,  48,  49,  50,  51,
          52,  53,  54,  55,  56,  57,  58,  59,  60,  61,  62,  63,  64,
          65,  66,  67,  68,  69,  70,  71,  72,  73,  74,  75,  76,  77,
          78,  79,  80,  81,  82,  83,  84,  85,  86,  87,  88,  89,  90,
          91,  92,  93,  94,  95,  96,  97,  98,  99, 100, 101, 102, 103,
         104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116,
         117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129,
         130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142,
         143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155,
         156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168,
         169, 170,

### Map which frames correspond to which frame between the two videos

In [15]:
print(path[0])  # Indices in bob
print(path[1])  # Indices in bobby

[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17
  18  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35
  36  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53
  54  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71
  72  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89
  90  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107
 108 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125
 126 127 128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143
 144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159 160 161
 162 163 164 165 166 167 168 169 170 171 172 173 174 175 176 177 178 179
 180 181 182 183 184 185 186 187 188 189 190 191 192 193 194 195 196 197
 198 199 200 201 202 203 204 205 206 207 208 209 210 211 212 213 214 215
 216 217 218 219 220 221 222 223 224 225 226 227 228 229 230 231 232 233
 234 235 236 237 238 239 240 241 242 243 244 245 24

### Calculate per-feature differences. Used to create data which can be used to find the specific deviations between videos (eg not deep enough squat based on not low enough knee angle)

In [16]:
idx_a, idx_b = path

aligned_diffs = []

for i, j in zip(idx_a, idx_b):
    diff = bob[i] - bobby[j]   # signed difference
    aligned_diffs.append(diff)

aligned_diffs = np.array(aligned_diffs)
# shape: (path_length, D)
aligned_diffs

array([[123.15653049, 122.40842037,  -1.99135315, ...,   4.43471576,
        -12.90151465,  60.91802851],
       [114.68702646, 109.12272609,  -1.6381347 , ...,  -0.81792063,
        -15.01262835,  61.42807481],
       [109.53977715, 100.11802712,  -1.43978206, ...,   1.17865169,
        -16.26811953,  61.13384011],
       ...,
       [ 83.2270684 ,  70.21714301,   2.12587145, ...,  -0.79668221,
         -4.43125479,  12.48261218],
       [ 83.07166822,  70.51626047,   2.08020039, ...,  -0.8732501 ,
         -4.42576707,  12.90552814],
       [ 82.95399257,  70.76322395,   2.13114651, ...,  -0.98858566,
         -5.03722017,  12.25264916]])

### Find which feature contribute most to the error

In [17]:
feature_error = np.mean(np.abs(aligned_diffs), axis=0)

feature_error

array([106.24140851,  82.67373082,  12.30308143,  12.50506685,
        24.01239429,  15.64320966,  18.99026211,  36.09410802,
         8.53842473,  23.46847733,  40.1986387 ,  33.74508377,
         2.26200767,   2.36814134,   1.37299495,   1.27348627,
         1.44111531,   1.1765634 ,   1.236957  ,   1.92228646,
         4.36436291,   5.33899264,   3.26008738,   2.55158866,
        22.5183369 ,  27.43937129])

### Find out when the difference occurs. Can be used to plot the difference in time.

In [18]:
knee_diff_over_time = np.abs(
    aligned_diffs[:, [0, 1]]
).mean(axis=1)


In [19]:
peak_idx = np.argmax(knee_diff_over_time)
peak_idx

147

In [20]:
bob_frame = idx_a[peak_idx]
bobby_frame = idx_b[peak_idx]
bob_frame, bobby_frame

(147, 126)

In [21]:
bob[:, 2]

array([177.90296683, 178.32625343, 178.5481822 , 178.59405598,
       178.48917759, 178.25884985, 177.87241058, 177.41785491,
       177.01418801, 176.75223373, 176.49644115, 176.31542009,
       176.23081954, 176.29709304, 176.42755805, 176.53032245,
       176.63104119, 176.71767839, 176.84362005, 176.98723173,
       177.02217741, 176.9746365 , 176.86029876, 176.72624942,
       176.54793453, 176.36916288, 176.21083464, 176.14295797,
       176.14516321, 176.23379402, 176.36867224, 176.54531461,
       176.73392508, 176.90369773, 177.03489056, 177.17620156,
       177.30614843, 177.46805649, 177.60448404, 177.71193391,
       177.79355425, 177.86741557, 177.96506339, 178.09814223,
       178.30703827, 178.53220329, 178.71934166, 178.88699262,
       179.00544796, 179.03446399, 179.00208754, 178.94638407,
       178.91296329, 178.91048593, 178.93474734, 178.94157945,
       178.93119463, 178.93000706, 178.93747917, 178.93718423,
       178.93673219, 178.94955016, 178.97870011, 179.02